# 05 — Adverse-Action Reason Codes (Explainable Denials)

## Why This Notebook Exists

Under the **Equal Credit Opportunity Act (ECOA)** and **Regulation B**, when a lender denies credit it must tell the applicant the **specific principal reasons** for the decision. The CFPB's **Circular 2022-03** made clear that this applies *equally to machine-learning models* — there is no "black-box exception." A model that cannot produce specific, accurate denial reasons is non-compliant, even if its outcomes are unbiased.

This notebook turns the PD model's per-applicant SHAP contributions into **consumer-friendly adverse-action reason codes** — the ranked, plain-language factors that pushed an applicant's risk up.

> **Scope note.** A full fair-lending program also performs *disparate-impact testing* (comparing approval rates across protected classes such as race, sex, and age) and searches for less-discriminatory alternatives. The public Lending Club dataset contains **no protected-class attributes**, so a meaningful disparate-impact test is not possible here, and we deliberately do not fabricate one. This notebook focuses on the piece the data genuinely supports: explainable, ECOA-style adverse-action notices.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd

from src import config as C
from src import data as D
from src import fairness as FA
from src.features import clean_frame, split_X_y

model = joblib.load(C.MODELS_DIR / "pd_model.pkl")
df = clean_frame(D.load_resolved(n=20000))
X, y = split_X_y(df)
print("Loaded PD model and", len(X), "applicants")

Loaded PD model and 20000 applicants


## 1. Reason Codes for a Single Applicant

For one applicant we compute the SHAP contributions of each feature to their predicted PD, keep the factors that *increase* risk, and translate them into standardized reason-code language.

In [2]:
# Pick a higher-risk applicant to illustrate a denial
pd_scores = model.predict_proba(X)[:, 1]
idx = int(np.argsort(pd_scores)[-50])  # a high-risk applicant
applicant = X.iloc[[idx]]
print(f"Predicted PD: {pd_scores[idx]:.1%}")

codes = FA.reason_codes(model, applicant, top_n=4)
for i, c in enumerate(codes, 1):
    print(f"  {i}. {c['reason']}  (contribution {c['contribution']:+.3f})")

Predicted PD: 55.2%


  1. Requested loan term carries elevated risk  (contribution +0.684)
  2. Debt-to-income ratio too high  (contribution +0.191)
  3. Number of open credit lines outside policy  (contribution +0.179)
  4. Public derogatory record on file  (contribution +0.133)


Each line is a principal reason a compliance team could place on an adverse-action notice. The reason text is mapped from the raw feature to consumer-friendly language (e.g. `dti` → "Debt-to-income ratio too high"), and the contribution is the SHAP value showing how strongly that factor raised the applicant's risk.

## 2. Batch Reason Codes & Portfolio Frequency

Running the reason-code generator across many denied applicants shows which factors most commonly drive denials — useful for fair-lending monitoring and for spotting an over-reliance on any single factor.

In [3]:
# Top reasons across the riskiest applicants (would-be denials)
denied_idx = np.argsort(pd_scores)[-500:]
all_reasons = []
for i in denied_idx[::5]:  # subsample for speed
    for c in FA.reason_codes(model, X.iloc[[int(i)]], top_n=3):
        all_reasons.append(c["reason"])
freq = pd.Series(all_reasons).value_counts()
print("Most common adverse-action reasons among high-risk applicants:")
display(freq.to_frame("count"))

Most common adverse-action reasons among high-risk applicants:


,count
Requested loan term carries elevated risk,99
Debt-to-income ratio too high,54
Revolving credit utilization too high,29
Requested loan amount too high,23
Revolving balance too high,22
Home-ownership status carries elevated risk,21
Income too low for the requested amount,19
Too many recent credit inquiries,12
Number of open credit lines outside policy,9
Public derogatory record on file,7


## 3. Compliance Summary

- Every model decision can be accompanied by **specific, ranked, plain-language reasons** derived directly from the model — satisfying ECOA / Reg B and CFPB Circular 2022-03.
- The reasons are **faithful to the model** (SHAP attributions of the actual prediction), not generic boilerplate.
- **Limitation, stated honestly:** without protected-class attributes in the dataset, formal disparate-impact testing and least-discriminatory-alternative search cannot be performed here. In a production setting these would run on internal application data with demographic proxies (e.g. BISG) under a documented compliance-management system.